In [2]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,T5ForConditionalGeneration,TrainingArguments

In [3]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

In [6]:
#random sampling
train_data=train_data.sample(n=8000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=800,random_state=42).reset_index(drop=True)

In [7]:
train_data.shape

(8000, 3)

In [8]:
val_data.shape

(800, 3)

### DATA PREPROCESSING

In [10]:
import re

In [11]:
def clean_data(text):
    text = re.sub(r"\r\n"," ",text) #lines
    text = re.sub(r"\s+"," ",text) #spaces
    text = re.sub(r"<.*?>"," ",text) #html tags
    text = text.strip().lower()

    return text

In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [13]:
train_data.sample(5)

,id,dialogue,summary
4467,13816487,april: have you got admission in aimc larry: i...,april is still waiting for her results regardi...
4529,13821819,tanya: we're leaving ireland tomorrow :( lexi:...,tanya is leaving ireland tomorrow.
946,13731198,"peter: hey maniac, have you seen blinded by th...","andrew watched ""blinded by the lights by zulcz..."
7262,13829205,allyn: so where are we going tonight? pricilla...,allyn and pricilla are going to the onetwofree...
2885,13729693,tess: what should i bring to the party? kaila:...,"tess should bring some cookies to the party, n..."


### TOKENIZE

In [15]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [16]:
# raw data -> tokeinze

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to i/p as labels
    return inputs

In [17]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [18]:
type(train_dataset)
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [19]:
# input ids - dialogue => token ids

# 1 => End oF sequence 0=> padding

#attention mask
# labels - target => summary token

### Working with the Model

In [ ]:
# NLP => generation task
!pip install ipywidgets
model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [22]:

import torch

if torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

print("Device: ",device)

Device:  cuda


In [23]:
# Training Arguments

training_args=TrainingArguments(
    output_dir="./results_8k",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500 #0 => learning rate towards the default value
)

In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Re-initialize the trainer with the new dataset and the new output_dir
# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=tokenized_train_dataset, # Now using your 8k data
#     eval_dataset=tokenized_val_dataset,
#     tokenizer=tokenizer,
# )

# Start training from scratch
# trainer.train()

In [25]:
print(f"Total samples in dataset: {len(train_dataset)}")

Total samples in dataset: 8000


### TRAINING THE MODEL

In [47]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.402996,0.351733
2,0.383982,0.339241
3,0.370644,0.333698
4,0.351790,0.332257
5,0.351074,0.331082
6,0.345439,0.330374


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6000, training_loss=0.6387778600056966, metrics={'train_runtime': 1936.3237, 'train_samples_per_second': 24.789, 'train_steps_per_second': 3.099, 'total_flos': 6496406470656000.0, 'train_loss': 0.6387778600056966, 'epoch': 6.0})

In [49]:
# model load => fine-tune => save the model

In [51]:
model.save_pretrained("./saved_summary_model_8k")
tokenizer.save_pretrained("./saved_summary_model_8k")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model_8k\\tokenizer_config.json',
 './saved_summary_model_8k\\tokenizer.json')

In [53]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model_8k/")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model_8k/")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
!pip install ipywidgets

### TEST THE CORE LOGIC FOR SUMMARISATION

In [55]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def summarize_dialogue(dialogue):
    dialogue=clean_data(dialogue) # CLEAN

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )

    inputs = inputs.to(device)
    
    # generate the summary => token ids
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    # token ids convert to summary => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # End of Seq, Tab spaces
    return summary

In [57]:
dialogue_1 = """
Manager: Hi team, we need to finalize the project deadline. 
John: I think we need at least two more weeks. 
Manager: That is not possible. We have a hard deadline for the client. 
John: Okay, I will try to expedite the testing phase. 
Manager: Thanks, John. Let's meet tomorrow at 10 AM to discuss the plan.
"""

dialogue_2 = """
Alice: Hey! Are you still coming over for dinner tonight?
Bob: Oh, I completely forgot! Is it okay if I come a bit late? I have a meeting that might run over.
Alice: That's fine, but please bring some dessert if you can!
Bob: Sure thing, I'll grab some ice cream on my way. See you! 
"""

dialogue_3 = """
Customer: Hi, I've been trying to log into my account for an hour, but it keeps showing an error.
Support: I am sorry to hear that. Could you please clear your browser cache and try again?
Customer: I already tried that, but it didn't help.
Support: I see. I'll escalate this to our technical team and get back to you shortly.
Customer: Thanks, I appreciate your help.
"""

# Create a list of dialogues
dialogues = [dialogue_1, dialogue_2, dialogue_3]

# Loop through and summarize each one separately
for i, d in enumerate(dialogues):
    print(f"--- Summary of Dialogue {i+1} ---")
    print(summarize_dialogue(d))
    print("\n")

--- Summary of Dialogue 1 ---
john and manager will meet tomorrow at 10 am to discuss the plan.


--- Summary of Dialogue 2 ---
bob forgot to come over for dinner tonight. alice will grab some ice cream on his way.


--- Summary of Dialogue 3 ---
customer has been trying to log into his account for an hour. support will escalate this to our technical team and get back to customer shortly.




In [93]:

#trainer.train(resume_from_checkpoint="./results/checkpoint-3000")

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss


TrainOutput(global_step=3000, training_loss=0.0, metrics={'train_runtime': 0.0021, 'train_samples_per_second': 11581143.12, 'train_steps_per_second': 1447642.89, 'total_flos': 3248203235328000.0, 'train_loss': 0.0, 'epoch': 6.0})